### 크롤링으로 웹 페이지 정보를 수집하기전 주의사항
* `robots.txt` 파일 확인해서 수집을 허용하는지 확인
* 이용약관에서 자동 수집 금지, 상업적 이용 금지, 재배포 금지 조항을 확인
* 요청 간격을 적절히 지정 

### 크롤링 과정
1. requests로 화면 html 코드를 가져옴
2. BS로 원하는 요소를 탐색
3. 탐색한 요소에 있는 정보를 추출
4. 활용(DataFrame)

## 1.requests로 화면 html 코드를 가져옴

In [1]:
import requests as req

BASE_URL = "https://kin.naver.com/"

def fetch_kin_html(query:str, page:int=1) -> str | None:
    try:
        result = req.get(f"{BASE_URL}/search/list.naver", params={"query":query, "page":page}, timeout=5)
        print(result.url)
        result.raise_for_status()
        return result.text
    except req.Timeout:
        print("요청 시간 초과")
    except req.RequestException as e:
        print(f"요청 실패 : {e}")
    return None
html = fetch_kin_html("파이썬",1)
print(html.strip())

https://kin.naver.com//search/list.naver?query=%ED%8C%8C%EC%9D%B4%EC%8D%AC&page=1
<!DOCTYPE html>
<html lang="ko">
<head>
<meta name="referrer" contents="always">

<meta http-equiv="X-UA-Compatible" content="IE=edge">
<meta name="description" content="파이썬의 지식iN Q&amp;A 검색결과입니다. 궁금증을 해결하지 못했다면 지식iN &#039;질문하기&#039;를 해보세요."/>
<meta name="viewport" content="width=1024">

<meta name="msapplication-config" content="none" />






































<link rel="stylesheet" type="text/css" href="https://ssl.pstatic.net/static.kin/static/pc/20261027180343/css/min/common.css" />
<link rel="stylesheet" type="text/css" href="https://ssl.pstatic.net/static.kin/static/pc/20261027180343/css/min/components.css" />


	
	
		<link rel="stylesheet" type="text/css" href="https://ssl.pstatic.net/static.kin/static/pc/20261027180343/css/min/other.css">
	






<script>
	
	var pcDomain = "kin.naver.com";
	var pcDomainWithProtocol = "https://kin.naver.com";
	var pcKinServiceProtocol = "https";

	va

### 2.BS로 원하는 요소를 탐색
* 탐색하기 전 원하는 요소의 선택자를 확인
* soup = BeautifulSoup(html코드, "html.parser") : html코드를 html객체로 분석
* els = soup.select("선택자") : 선택자와 일치하는 모든 요소들. 리스트로 반환
* el = soup.select_one("선택자") : 선택자와 일치하는 첫번째 요소.
* el.get_text(strip=True) : 공백을 제거하고 텍스트를 가져옴
* el.get("속성명"), el["속성명"] : 요소의 속성명과 일치하는 속성의 값을 가져옴
 

In [2]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(html, "html.parser")

links = soup.select("._searchListTitleAnchor")
for link in links:
    print(link)

<a class="_nclicks:qna.txt _searchListTitleAnchor" href="https://kin.naver.com/qna/detail.naver?dirId=10402&amp;docId=495495173&amp;answerNo=2&amp;qb=7YyM7J207I2s" onclick="pushClick(nlogPageCode.kin_search_result,'kin_search_result_card',{content_id:'495495173',content_id_origin:'doc_id',content_source0:'10402',content_source0_origin:'dir_id',global_card_index:0});" target="_blank">정보 과목 <b>파이썬</b> 코드 질문</a>
<a class="_nclicks:qna.txt _searchListTitleAnchor" href="https://kin.naver.com/qna/detail.naver?dirId=10402&amp;docId=491026040&amp;answerNo=2&amp;qb=7YyM7J207I2s" onclick="pushClick(nlogPageCode.kin_search_result,'kin_search_result_card',{content_id:'491026040',content_id_origin:'doc_id',content_source0:'10402',content_source0_origin:'dir_id',global_card_index:1});" target="_blank"><b>파이썬</b> 아이콘만드는방법</a>
<a class="_nclicks:qna.txt _searchListTitleAnchor" href="https://kin.naver.com/qna/detail.naver?dirId=10402&amp;docId=495481671&amp;answerNo=1&amp;qb=7YyM7J207I2s" onclick="push

## 3. 탐색한 요소에 있는 정보를 추출

In [3]:
list = []
for link in links:
    # 제목
    title = link.get_text(strip=True)
    # 제목 링크
    href = link.get("href")
    list.append({"title" : title, "href" : href})

In [4]:
import pandas as pd
df = pd.DataFrame(list)
df.head(5)

,title,href
0,정보 과목파이썬코드 질문,https://kin.naver.com/qna/detail.naver?dirId=1...
1,파이썬아이콘만드는방법,https://kin.naver.com/qna/detail.naver?dirId=1...
2,"파이썬기초 질문(urllib, beautifulsoup4)",https://kin.naver.com/qna/detail.naver?dirId=1...
3,파이썬코딩 질문입니다.,https://kin.naver.com/qna/detail.naver?dirId=1...
4,파이썬리스트 슬라이싱,https://kin.naver.com/qna/detail.naver?dirId=1...


### df에 있는 href를 이용하여 질문 상세와 대표답변 1개를 가져오는 작업
###   검색어("파이썬")으로 등록된 질문과 질문 상세, 첫번째 답변을 크롤링하는 작업
###   결과물은 df에 title, title_detail, answer가 들어가도록 작업

In [7]:
# 질문 링크를 이용하여 질문 상세 페이지 html 코드 가져오기
href = df.iloc[0]["href"]
def fetch_kin_detail_html (href: str) -> str | None:
    try:
        result = req.get(href, timeout=5)
        result.raise_for_status()
        return result.text                     # 담기만 하지 말고 돌려주기
    except req.Timeout:
        print("요청 시간 초과")
    except req.RequestException as e:
        print(f"요청 실패 : {e}")
    return None

# href를 이용해서 질문 상세 페이지에 있는 html 코드 가져오기
html = fetch_kin_detail_html(href)            # 함수를 불러서 결과를 html에 담기

In [8]:
# bs를 이용해서 요소를 탐색(질문 상세, 첫번째 답변)
from bs4 import BeautifulSoup

soup = BeautifulSoup(html, "html.parser")

# 원하는 요소 탐색
detail_el= soup.select_one (".questionDetail")
answer_el= soup.select_one (".answerDetail")



In [9]:
# 요소에서 필요한 내용 추출(질문 상세, 첫번째 답변)
detail = detail_el.get_text(strip=True)
answer = answer_el.get_text(strip=True) if answer_el is not None else None
# print(detail)
# print(answer)

In [20]:
import time
from bs4 import BeautifulSoup

def fetch_kin_detail_html(href:str)->str | None:
    try:
        result = req.get(href, timeout=5)
        result.raise_for_status()
        return result.text                     # 담기만 하지 말고 돌려주기
    except req.Timeout:
        print("요청 시간 초과")
    except req.RequestException as e:
        print(f"요청 실패 : {e}")
    return None

def get_detail_answer(html:str)->dict | None:
    if html is None:
        return None
    soup = BeautifulSoup(html, "html.parser")     # 받은 html로 soup를 새로 만들기
    detail_el= soup.select_one (".questionDetail")
    answer_el= soup.select_one (".answerDetail")
    detail = detail_el.get_text(strip=True) if detail_el is not None else None
    answer = answer_el.get_text(strip=True) if answer_el is not None else None
    return { "detail" : detail, "answer" : answer}
    
for row in df.itertuples():
    href = row.href
    html = fetch_kin_detail_html(href)
    result = get_detail_answer(html)
    print(result)
    time.sleep(0.1)                 # 요청 간격 지정

{'detail': '이 두문제 풀이랑 답좀 알려주세요', 'answer': '두 문제 모두 차근차근 따라가 볼게요.\u200b1. 16번 (출력되는 줄 수)\u200bprint(\'코딩은\', \'재밌다\', end=\' \')→ 끝에 줄바꿈 대신 공백을 붙여서 줄이 바뀌지 않습니다.\u200bprint(\'그러나\', \'매우\', \'어렵다\', sep=\'@@\')→ 앞 줄에 이어서 출력되고 이번엔 줄바꿈합니다.1줄: 코딩은 재밌다 그러나@@매우@@어렵다\u200bprint(\'그래도\', \'열심히\', sep=\'\\n\')→ 값 사이에 줄바꿈이 들어갑니다.2줄: 그래도3줄: 열심히\u200bprint(\'어차피\', \'해야됨\')4줄: 어차피 해야됨\u200b그래서 정답은 4줄(③)입니다. 첫 print의 end=\' \' 때문에 첫 두 print가 한 줄로 합쳐지는 게 포인트예요.\u200b2. 12번 (리스트 인덱싱)\u200b인덱스는 0부터 셉니다.- SSD[4] = [3, 2] → SSD[4][1] = 2- HDD[2] = [7, 16] → HDD[2][1] = 16- SSD[1] = [2, 1] → SSD[1][1] = 1\u200bend=\' \'라서 "2 16 1"이 한 줄에 출력되고, 선지에서는 공백을 빼고 2161로 나타냈으니 정답은 ①입니다.\u200b②를 고르셨다면 HDD[2]를 [22, 15]로 보신 것 같아요. 두 번째 원소(인덱스 1)가 [22, 15], 세 번째(인덱스 2)가 [7, 16]입니다. 리스트 위에 0, 1, 2, 3 번호를 적어 두고 풀면 실수가 줄어요.'}
{'detail': '아이콘 어떻게만드나요?', 'answer': "해결 방법팟플레이어 오디오 트랙 변경팟플레이어로 영상 재생 중 Alt + A를 누르거나 마우스 우클릭 > [소리] > [오디오 트랙 선택]에서 다른 트랙(예: 트랙 2)으로 변경해 보세요.(확인: 트랙 전환 후 외부 잡음 대신 콘서트 오디오가 나오는지 확인)Game Bar 오디오 설정 

In [21]:
#list를 DataFrame으로 변환
df_answer = pd.DataFrame(list)
df_answer.head(3)

,title,href
0,정보 과목파이썬코드 질문,https://kin.naver.com/qna/detail.naver?dirId=1...
1,파이썬아이콘만드는방법,https://kin.naver.com/qna/detail.naver?dirId=1...
2,"파이썬기초 질문(urllib, beautifulsoup4)",https://kin.naver.com/qna/detail.naver?dirId=1...


In [22]:
df_merged = pd.merge(df, df_answer, left_index=True, right_index=True)
df_merged.head(3)

,title_x,href_x,title_y,href_y
0,정보 과목파이썬코드 질문,https://kin.naver.com/qna/detail.naver?dirId=1...,정보 과목파이썬코드 질문,https://kin.naver.com/qna/detail.naver?dirId=1...
1,파이썬아이콘만드는방법,https://kin.naver.com/qna/detail.naver?dirId=1...,파이썬아이콘만드는방법,https://kin.naver.com/qna/detail.naver?dirId=1...
2,"파이썬기초 질문(urllib, beautifulsoup4)",https://kin.naver.com/qna/detail.naver?dirId=1...,"파이썬기초 질문(urllib, beautifulsoup4)",https://kin.naver.com/qna/detail.naver?dirId=1...


In [28]:
import requests as req

def fetch_kin_html(query:str, page:int=1) -> str | None:
    BASE_URL = "https://kin.naver.com"
    try:
        # params의 역할(get) : url 뒤에 ?변수=값형태로 변환
        # ?query=값&page=값
        result = req.get(f"{BASE_URL}/search/list.naver", params={"query":query, "page":page}, timeout=5)
        # print(result.url) # 확인용
        result.raise_for_status() # 정상응답이 아니면 예외를 발생시킴
        return result.text # 응답 결과에서 문자들을 반환(html 코드)
    except req.Timeout:
        print("요청 시간 초과")
    except req.RequestException as e:
        print(f"요청 실패 : {e}")
    return None

In [25]:
from bs4 import BeautifulSoup
def get_question_list(html: str) -> list | None:
    soup = BeautifulSoup(html, "html.parser")
    links = soup.select("._searchListTitleAnchor")
    if len(links) == 0:          # select는 못 찾으면 빈 리스트 []
        return None
    list = []
    for link in links:
        title = link.get_text(strip=True)
        href = link.get("href")
        list.append({"title" : title, "href" : href})
    return list

In [26]:
def get_question_df(list:list[dict]|None)->pd.DataFrame|None:
    if list is None:
        return None
    return pd.DataFrame(list)

In [29]:
import time

def fetch_kin_detail_html(href:str)->str|None:
    try:
        result = req.get(href, timeout=5)
        result.raise_for_status() # 정상응답이 아니면 예외를 발생시킴
        return result.text # 응답 결과에서 문자들을 반환(html 코드)
    except req.Timeout:
        print("요청 시간 초과")
    except req.RequestException as e:
        print(f"요청 실패 : {e}")
    return None

def get_detail_answer(html:str|None)->dict|None:
    if html is None:
        return None
    soup = BeautifulSoup(html, "html.parser") #html코드로 된 문자열을 탐색이 쉽게 만들어줌
    # 원하는 요소를 탐색
    detail_el = soup.select_one(".questionDetail")
    answer_el = soup.select_one(".answerDetail")
    detail = detail_el.get_text(strip=True)
    answer = answer_el.get_text(strip=True) if answer_el is not None else None # 답변이 등록 안된 경우는 None으로 
    return { "detail" : detail, "answer" : answer}

In [30]:
def get_answer_df(df:pd.DataFrame | None)->pd.DataFrame|None:
    if df is None:
        return None
    list = []    
    for row in df.itertuples():
        href = row.href
        html = fetch_kin_detail_html(href)
        result = get_detail_answer(html)
        list.append(result)
        time.sleep(0.1) # 너무 빠른 시간에 많이 요청하면 ip차단될수 있어서 지연요청
    return pd.DataFrame(list)

In [31]:
def crawling_kin(query:str, max_page:int = 1)->pd.DataFrame:
    if max_page <= 0:
        max_page = 1
    df_total = pd.DataFrame(columns=["title", "href", "detail", "answer"])
    for page in range(1, max_page+1):
        html = fetch_kin_html(query, page)
        list = get_question_list(html)
        df = get_question_df(list)
        df_answer = get_answer_df(df)
        df_merged = pd.merge(df, df_answer, left_index=True, right_index=True)
        df_total = pd.concat([df_total, df_merged], ignore_index=True)
        print(f"{page}/{max_page} 크롤링 완료")
    return df_total

In [32]:

df = crawling_kin("파이썬", 2)
print(df.shape)
# df

1/2 크롤링 완료
2/2 크롤링 완료
(20, 4)
